# 第4章 DC モータ：PWM・エンコーダ・分割区法（ノートブック版）

教科書の第4章のうち、**Pico B とモータがなくても動く Mac 側の解析**（4-6 の `motor_analysis.py`・`pole_period.py`・`split_plot.py`）を、本文の合成データ `synth_motor.py` で最後まで試せるように1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約40秒）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | デューティのスイープ：動き出すデューティ、回転数が比例しないこと（不連続導通） | 手順 7〜9（C）、4-3-8 | `synth_motor.py sweep` → `motor_analysis.py sweep` |
| 2 | ステップ応答：差分＋curve_fit と、状態空間＋カルマンフィルタで τ・K | 手順 11（D） | `synth_motor.py step` → `motor_analysis.py step` |
| 3 | PWM の周波数を変えたときの回転数 | 手順 12（E） | `synth_motor.py sweep --freq` → `motor_analysis.py sweep` |
| 4 | ロジックアナライザの記録から P（1回転のパルス）と減速比 G | 手順 13（F） | `synth_motor.py la` → `pole_period.py` |
| 5 | 分割区法の要因計画（V×L×デューティ）：誤った分析と正しい分析 | 手順 14（G） | `split_plot.py design` → `synth_motor.py fact` → `split_plot.py analyze` |
| 6 | デューティ 100% の RUN だけで R・K_e（dcfit）と、全部の RUN で当てはめる誤り | 手順 14（G）、4-7 | `motor_analysis.py dcfit`（比べの当てはめはノートブックで追加） |

- 次のセルたちが、4-6 の Mac 側の解析のファイル4つを、このノートブックの隣に `%%writefile` で書き出します（教科書の `~/home-lab/ch4/` の代わり）。中身は 4-6 と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- 実機の記録の代わりに、本文の `synth_motor.py` の合成データ（真の値 $P = 7$、$G = 98$ → CPR_out = 2744、$\tau = 40$ ms、$R = 8\,\Omega$、$K_e = 0.161$ V·s/rad、摩擦の電流 60 mA、コイルの $L/R = 50\,\mu$s、D1 の順電圧 0.3 V、不連続導通あり、電流は INA219 と同じく電池側の値）を使います。そのため `--cpr-out` は本文の手順の 2800（書き方の例）ではなく **2744** にします。データは `data/` ではなく `data_synth/` に作ります（本文の `setup_ch4.sh` と同じ）。
- 各節では、ターミナルの `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（中では `python ○○.py …` を別のプロセスで動かします）。表示を見てから、`ok()` で 4-7 の「結果の例」の合成データの数値と、合成の真の値とを比べて確かめます。
- 乱数の種は本文の例と同じにしてあります（`synth_motor.py` の `--seed` は既定の 0、`split_plot.py design --seed 0`）。この設定で、4-7 の合成データの数値（49%・82%・95%、τ = 43.4・38.7 ms、248.1・243.8・222.8 rpm、L = 7・r = 0.989、F = 77.5・30.7、R = 8.11 Ω など）がそのまま出ることを確かめてあります。numpy・scipy・statsmodels の版で最後の桁が変わることがあります。
- **入れていないもの（Pico B とモータが要る部分）**：回路を組むこと、`motor.c`・`quadrature.pio`（C SDK。Pico B に書き込むプログラム）、`motor_run.py` でのシリアルの記録（手順 1〜14 の実物の部分）、テスタでの電圧の確かめ、Pi の `sigrok-cli` での記録。`motor_run.py` は下に読むためだけに載せてあります（実行しません）。自分の記録（`motor_run.py` の CSV）を Colab の左のファイルの欄に上げれば、同じ `run("motor_analysis.py sweep ○○.csv --cpr-out 自分の値")` で解析できるはずですが、執筆時には確かめていません。
- ⚡ 実物の注意（電池を入れる前の配線の確かめ、非常停止、Q1・D1・C1 の向き、ポリスイッチ）は、教科書の 4-4・4-5 を読んでください。このノートブックは計算だけです。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels が入っている。版は要確認）
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(_v, "1")   # 小さい行列の計算だけなので1スレッドにする（CPU が混んでいると、多スレッドはかえって遅くなる）
import importlib, subprocess, sys
for mod, pkg in [("statsmodels", "statsmodels"), ("pandas", "pandas"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import statsmodels, scipy, numpy, pandas, matplotlib
print("OK", "numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__,
      "statsmodels", statsmodels.__version__, "matplotlib", matplotlib.__version__)
print("（ノートブックは numpy 2.5.5・scipy 1.18.1・pandas 3.1.1・statsmodels 0.15.0・matplotlib 3.11.2 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

## Pico B と記録の側（読むだけ。実行しません）

教科書の手順では、Pico B に `motor.c`＋`quadrature.pio`（C SDK）を書き込み、Mac の `motor_run.py` がシリアルで `RUN`・`SWEEP` などのコマンドを送って、1 ms ごとの行（`t_ms,duty,count,ina_seq,i_ma,v_bus`）を CSV に保存します。`count` は PIO が 4 逓倍で数えたエンコーダのカウント、`i_ma`・`v_bus` は INA219（約 9 ms ごとに新しい値になり、そのとき `ina_seq` が 1 増える）の電池側の電流と電圧です。スイープの CSV には、さらに `run_id`・`duty_set`・`rep`・`host_time` の列が付きます。`synth_motor.py` は、これと同じ形の CSV を作ります。

`motor_run.py` を、そのまま載せておきます（Colab では Pico につなげないので、実行しません。C のプログラムは教科書の 4-6 を見てください）。

**Mac｜~/home-lab/ch4/motor_run.py**（読むだけ）

```python
#!/usr/bin/env python3
"""第4章：Pico B（motor.c）にコマンドを送り、モータを決まった時間だけ回して記録する

使い方（~/home-lab/ch4 で）:
  uv run python motor_run.py ping
  uv run python motor_run.py count                 # 出力軸を手で N 回まわす（Ctrl-C で終わる）
  uv run python motor_run.py run --d1 500 --ms1 3000 --out data/run_d500.csv
  uv run python motor_run.py sweep --ms 3000 --rep 2 --order random --seed 1 --out data/sweep_1.csv
  uv run python motor_run.py step --lo 300 --hi 700 --ms 1500 --n 5 --out data/step_1.csv
  共通: --freq 1000（PWM の周波数 [Hz]）、--port /dev/cu.usbmodemXXXX
デューティは千分率（0〜1000。500 = 50%）。1回の RUN は合計60秒まで（Pico B の側でも制限している）。
"""
import argparse
import csv
import datetime
import glob
import json
import pathlib
import random
import sys
import time

import serial

FIELDS = ["t_ms", "duty", "count", "ina_seq", "i_ma", "v_bus"]


def open_port(port):
    ports = sorted(glob.glob("/dev/cu.usbmodem*"))
    if not ports and not port:
        sys.exit("Pico B のシリアルポートが見つかりません")
    s = serial.Serial(port or ports[0], 115200, timeout=1)
    time.sleep(0.3)
    s.reset_input_buffer()
    s.write(b"PING\n")
    for _ in range(5):
        if s.readline().decode(errors="ignore").strip() == "PONG":
            return s
    sys.exit("PONG が返りません。Pico B に motor の UF2 が書き込まれているか、Pico A のポートを開いていないか確かめてください")


def command(s, text, prefix, timeout=3.0):
    """1行のコマンドを送り、prefix で始まる行が返るまで待つ"""
    s.write((text + "\n").encode())
    t0 = time.time()
    while time.time() - t0 < timeout:
        ln = s.readline().decode(errors="ignore").strip()
        if ln.startswith("E,"):
            sys.exit(f"Pico B がエラーを返しました：{ln}")
        if ln.startswith(prefix):
            return ln
    sys.exit(f"「{text}」への返事（{prefix}…）が来ません")


def set_freq(s, freq):
    ln = command(s, f"FREQ {freq}", "I,freq")
    return float(ln.split(",")[2])


def run_once(s, d1, ms1, d2=None, ms2=None):
    """RUN を1回行い、D 行を辞書のリストで返す"""
    cmd = f"RUN {d1} {ms1}" + (f" {d2} {ms2}" if d2 is not None else "")
    total_ms = ms1 + (ms2 or 0)
    s.write((cmd + "\n").encode())
    rows, t_last = [], time.time()
    try:
        return _collect(s, rows, t_last, total_ms)
    except KeyboardInterrupt:                     # Ctrl-C：まずモータを止める。記録の扱いは呼んだ側で決める
        s.write(b"STOP\n")
        t0 = time.time()                          # 届いている D 行を読み捨てながら、I,stop を待つ（最大3秒）
        while time.time() - t0 < 3.0:
            ln = s.readline().decode(errors="ignore").strip()
            if ln.startswith("I,stop"):
                break
        print("\nCtrl-C：STOP を送りました")
        raise


def _collect(s, rows, t_last, total_ms):
    while True:
        raw = s.readline()
        ln = raw.decode(errors="ignore").strip()
        if not ln:
            if time.time() - t_last > 5 + total_ms / 1000:
                s.write(b"STOP\n")
                sys.exit("データが来ません。STOP を送りました")
            continue
        t_last = time.time()
        if ln.startswith("D,"):
            v = ln.split(",")[1:]
            if len(v) != len(FIELDS):
                continue                          # 壊れた行はとばす（数を最後に表示）
            r = dict(zip(FIELDS, v))
            r["host_time"] = f"{time.time():.3f}"
            rows.append(r)
        elif ln.startswith("I,done"):
            _, _, n, dropped = ln.split(",")
            n, dropped = int(n), int(dropped)
            if dropped > 0:
                print(f"  ⚠ Pico B の側で {dropped} 行を取りこぼしました（USB が詰まった）。記録は使えるが時刻に穴がある")
            if n != len(rows):
                print(f"  ⚠ Pico B は {n} 行送ったが、受け取れたのは {len(rows)} 行")
            return rows
        elif ln.startswith("I,undervolt"):
            print(f"  ⚠ V_M が下がったので Pico B がモータを止めました（{ln}）。非常停止・電池を確かめる")
        elif ln.startswith("E,"):
            if "INA219" in ln:
                sys.exit(f"Pico B がエラーを返しました：{ln}（INA219 が応答しないと V_M の保護が効かないので RUN しない。I2C の配線を確かめる）")
            sys.exit(f"Pico B がエラーを返しました：{ln}")


def check_out(path):
    out = pathlib.Path(path)
    if out.exists():
        sys.exit(f"{out} はすでにあります（上書きしないように止めました）")
    out.parent.mkdir(parents=True, exist_ok=True)
    return out


def save(out, rows, extra_cols, meta):
    cols = extra_cols + FIELDS + ["host_time"]
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)
    out.with_suffix(".json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
    print(f"{out}（{len(rows)} 行）と {out.with_suffix('.json').name} を保存しました")


def stop_early(out, rows, extra_cols, meta):
    """sweep・step の途中の Ctrl-C：止めた RUN は捨て、それまでに終わった RUN だけを保存する"""
    if not rows:
        print("終わった RUN がないので、何も保存しません")
        return
    meta["interrupted"] = True
    print("途中で止めた RUN は捨て、それまでに終わった RUN だけを保存します（JSON に interrupted: true）")
    save(out, rows, extra_cols, meta)


def summary(rows, label):
    if not rows:
        return
    c0, c1 = int(rows[0]["count"]), int(rows[-1]["count"])
    i = max(float(r["i_ma"]) for r in rows)
    print(f"  {label}：{len(rows)} 行、カウント {c1 - c0:+d}、電流の最大 {i:.0f} mA、電源 {float(rows[-1]['v_bus']):.2f} V")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", default=None)
    sub = ap.add_subparsers(dest="cmd", required=True)
    sub.add_parser("ping")
    sub.add_parser("count")
    p = sub.add_parser("run")
    p.add_argument("--d1", type=int, required=True)
    p.add_argument("--ms1", type=int, required=True)
    p.add_argument("--d2", type=int, default=None)
    p.add_argument("--ms2", type=int, default=None)
    p.add_argument("--freq", type=float, default=1000)
    p.add_argument("--out", required=True)
    p = sub.add_parser("sweep")
    p.add_argument("--duties", default=",".join(str(d) for d in range(0, 1001, 100)))
    p.add_argument("--ms", type=int, default=3000)
    p.add_argument("--rep", type=int, default=2)
    p.add_argument("--order", choices=["random", "up"], default="random", help="random＝くり返しごとにでたらめな順、up＝--duties に書いた順")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--freq", type=float, default=1000)
    p.add_argument("--out", required=True)
    p = sub.add_parser("step")
    p.add_argument("--lo", type=int, default=300)
    p.add_argument("--hi", type=int, default=700)
    p.add_argument("--ms", type=int, default=1500)
    p.add_argument("--n", type=int, default=5)
    p.add_argument("--freq", type=float, default=1000)
    p.add_argument("--out", required=True)
    a = ap.parse_args()

    out = check_out(a.out) if hasattr(a, "out") else None
    s = open_port(a.port)
    meta = {"args": vars(a), "date": datetime.datetime.now().isoformat(timespec="seconds"), "port": s.port}

    if a.cmd == "ping":
        print("PONG が返りました")
        return
    if a.cmd == "count":
        command(s, "ZERO", "I,zero")
        print("カウントを0にしました。出力軸を手でまわしてください（Ctrl-C で終わる）")
        last = 0
        try:
            while True:
                last = int(command(s, "COUNT", "I,count").split(",")[2])
                print(f"\r  count = {last:+8d}", end="", flush=True)
                time.sleep(0.5)
        except KeyboardInterrupt:
            print(f"\n最後のカウント：{last:+d}（まわした回数で割ると、出力軸1回転あたりのカウント CPR_out）")
        return

    meta["freq_actual"] = set_freq(s, a.freq)
    print(f"PWM の周波数：{meta['freq_actual']:.1f} Hz")
    if a.cmd == "run":
        try:
            rows = run_once(s, a.d1, a.ms1, a.d2, a.ms2)
        except KeyboardInterrupt:
            sys.exit("途中で止めた RUN の記録は保存しません")
        summary(rows, "RUN")
        save(out, rows, [], meta)
    elif a.cmd == "sweep":
        duties = [int(d) for d in a.duties.split(",")]
        if any(d < 0 or d > 1000 for d in duties):
            sys.exit("デューティは 0〜1000 で指定してください")
        plan = []
        rng = random.Random(a.seed)
        for rep in range(a.rep):
            ds = duties[:]
            if a.order == "random":
                rng.shuffle(ds)                   # くり返しごとに順番を変える（電池の消耗などと交絡させない）
            plan += [(d, rep) for d in ds]
        meta["plan"] = plan
        print(f"{len(plan)} 回の RUN（各 {a.ms} ms）。合計 約{len(plan) * (a.ms / 1000 + 1.5):.0f} 秒")
        rows = []
        try:
            for k, (d, rep) in enumerate(plan):
                r = run_once(s, d, a.ms)
                summary(r, f"[{k + 1}/{len(plan)}] duty {d}")
                for x in r:
                    x.update(run_id=k, duty_set=d, rep=rep)
                rows += r
                time.sleep(1.0)
        except KeyboardInterrupt:
            stop_early(out, rows, ["run_id", "duty_set", "rep"], meta)
            return
        save(out, rows, ["run_id", "duty_set", "rep"], meta)
    elif a.cmd == "step":
        rows = []
        try:
            for k in range(a.n):
                r = run_once(s, a.lo, a.ms, a.hi, a.ms)
                summary(r, f"[{k + 1}/{a.n}] {a.lo}→{a.hi}")
                for x in r:
                    x.update(run_id=k, lo=a.lo, hi=a.hi, ms=a.ms)
                rows += r
                time.sleep(1.0)
        except KeyboardInterrupt:
            stop_early(out, rows, ["run_id", "lo", "hi", "ms"], meta)
            return
        save(out, rows, ["run_id", "lo", "hi", "ms"], meta)


if __name__ == "__main__":
    main()
```

次の4つのセルで、解析のファイルと合成データのファイルを書き出します（`split_plot.py` は、同じフォルダの `motor_analysis.py` の `summarize_sweep` を読み込みます）。

In [ ]:
%%writefile motor_analysis.py
#!/usr/bin/env python3
"""第4章：モータの記録（motor_run.py の CSV）を解析する

使い方（~/home-lab/ch4 で）:
  uv run python motor_analysis.py sweep data/sweep_1.csv --cpr-out 2744 [--gear 98] [--table data/sweep_1_table.csv]
      デューティと回転数・電池側の電流の関係。動き出すデューティ u0、全速との比（比例とのずれ）
  uv run python motor_analysis.py dcfit data/fact/S0*.csv --cpr-out 2744
      デューティ 100% の RUN だけで DC モータの式 V_bus = R·I + Ke·ω を回帰（電源電圧・負荷を変えたものを集める）
  uv run python motor_analysis.py step data/step_1.csv --cpr-out 2744 [--run 0]
      ステップ応答から時定数 τ とゲイン K を求める（差分＋非線形最小二乗、状態空間モデル＋カルマンフィルタ）
  --cpr-out : 出力軸1回転あたりのカウント（4逓倍。手で回して求めた値）
"""
import argparse
import sys

import numpy as np
import pandas as pd
from scipy import optimize, stats

SKIP_MS = 1000          # 定常とみなすまで捨てる時間（τ の 10 倍以上）
TAIL_MS = 300           # RUN の後ろに Pico B が付ける「デューティ 0」の時間


def load(path):
    df = pd.read_csv(path)
    need = {"t_ms", "duty", "count", "ina_seq", "i_ma", "v_bus"}
    if not need <= set(df.columns):
        sys.exit(f"{path} の列が足りません：{sorted(need - set(df.columns))}")
    return df


def steady(g, cpr_out):
    """1回の RUN（同じ run_id）の定常部分から、回転数・電流・電圧の平均を求める"""
    t_end = g["t_ms"].max() - TAIL_MS
    s = g[(g["t_ms"] >= SKIP_MS) & (g["t_ms"] < t_end)]
    if len(s) < 100:
        return None
    slope = np.polyfit(s["t_ms"], s["count"], 1)[0]             # カウント/ms
    rpm = slope * 60000 / cpr_out
    ina = s.drop_duplicates("ina_seq")                           # INA219 の新しい値だけ
    return {"rpm": rpm, "i_ma": ina["i_ma"].mean(), "v_bus": ina["v_bus"].mean(), "n_ina": len(ina)}


def summarize_sweep(df, cpr_out):
    out = []
    for rid, g in df.groupby("run_id"):
        r = steady(g, cpr_out)
        if r is None:
            continue
        r.update(run_id=rid, duty_set=int(g["duty_set"].iloc[0]),
                 rep=int(g["rep"].iloc[0]) if "rep" in g else 0)
        out.append(r)
    return pd.DataFrame(out)


def cmd_sweep(a):
    df = load(a.csv)
    t = summarize_sweep(df, a.cpr_out)
    if a.gear:
        t["rpm_motor"] = t["rpm"] * a.gear
    t = t.sort_values(["duty_set", "rep"])
    pd.set_option("display.width", 120)
    print("RUN ごとの定常値（最初の 1 秒を捨て、カウントの傾きから回転数）")
    print(t.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    if a.table:
        t.to_csv(a.table, index=False)
        print(f"{a.table} を保存しました")

    if t["duty_set"].nunique() < 3:             # 周波数の比較など、デューティが1〜2種類のとき
        g = t.groupby("duty_set")["rpm"]
        for duty, s in g:
            se = s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else float("nan")
            print(f"\nデューティ {duty:.0f}‰：回転数の平均 {s.mean():.2f} rpm（標準誤差 {se:.2f}、{len(s)} RUN）")
        print("デューティが3種類より少ないので、全速との比は出しません")
        if a.plot:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
            fig, ax = plt.subplots(figsize=(5, 4))
            ax.plot(t["duty_set"], t["rpm"], "o")
            ax.set_xlabel("デューティ [‰]")
            ax.set_ylabel("出力軸の回転数 [rpm]")
            ax.set_title("RUN ごとの回転数")
            fig.tight_layout()
            fig.savefig(a.plot, dpi=120)
            print(f"{a.plot} を保存しました")
        return
    d, y = t["duty_set"].to_numpy(float), t["rpm"].to_numpy()
    g = t.groupby("duty_set")["rpm"].agg(["mean", "std", "count"])
    still = g.index[g["mean"] < 5]                # 回らなかった（5 rpm 未満）デューティ
    moving = g.index[g["mean"] >= 5]
    if len(moving) == 0:
        sys.exit("どの RUN でも回っていません（配線・電池・非常停止を確かめる）")
    lo_d = max([x for x in still if x < moving.min()], default=None)
    print(f"\n動き出すデューティ u0：{lo_d if lo_d is not None else '?'}‰ と {moving.min()}‰ の間"
          "（細かく知りたいときは、その間を --duties で刻んで測る）")
    full = g["mean"].max()
    print("デューティごとの回転数（全速＝いちばん速いデューティの平均との比。比例なら 比＝デューティ）：")
    for duty, r in g.iterrows():
        print(f"  {duty:5.0f}‰：{r['mean']:7.1f} rpm（sd {r['std']:5.1f}、{int(r['count'])} RUN）"
              f"  全速の {r['mean'] / full * 100:5.1f}%（比例なら {duty / 10:5.1f}%）")
    print("  ※ 電流が PWM のオフの間に途切れる（不連続導通）と、回転数はデューティに比例せず、")
    print("     動き出してすぐ速くなり、そのあと頭打ちになる（比が「比例なら」より大きい）。")
    print("  ※ i_ma は INA219 が測った電池側の電流（Q1 の上流）。PWM のオフの間に D1 を回る電流は入らないので、")
    print("     デューティ 100% 以外ではモータの電流ではない。R・Ke は dcfit（デューティ 100% の RUN だけ）で求める。")

    if a.plot:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].plot(d, y, "o", label="RUN ごと")
        ax[0].plot([0, 1000], [0, full], "r--", label="比例（原点と全速を結ぶ）")
        ax[0].set_xlabel("デューティ [‰]")
        ax[0].set_ylabel("出力軸の回転数 [rpm]")
        ax[0].legend()
        ax[1].plot(t["duty_set"], t["i_ma"], "o")
        ax[1].set_xlabel("デューティ [‰]")
        ax[1].set_ylabel("電池側の電流 [mA]（INA219 の平均）")
        fig.tight_layout()
        fig.savefig(a.plot, dpi=120)
        print(f"{a.plot} を保存しました")


def cmd_dcfit(a):
    """デューティ 100%（PWM なし）の RUN だけで、V_bus = R·I + Ke·ω を当てはめる。
    100% なら電池側の電流＝モータの電流で、モータにかかる電圧＝V_bus（Q1 の電圧降下は無視）"""
    rows = []
    for path in a.csv:
        df = load(path)
        if "duty_set" not in df:
            df["duty_set"], df["run_id"] = df["duty"].max(), 0
        t = summarize_sweep(df, a.cpr_out)
        t = t[t["duty_set"] == 1000]
        for _, r in t.iterrows():
            rows.append(dict(file=path, run_id=r["run_id"], rpm=r["rpm"], i_ma=r["i_ma"], v_bus=r["v_bus"]))
    m = pd.DataFrame(rows)
    if len(m) < 3:
        sys.exit("デューティ 100% の RUN が3つ以上必要です（電源電圧や負荷を変えたものを集める）")
    m["i_a"] = m["i_ma"] / 1000
    m["omega"] = m["rpm"] * 2 * np.pi / 60       # 出力軸の角速度 [rad/s]
    pd.set_option("display.width", 120)
    print("デューティ 100% の RUN（1 RUN = 1 点）")
    print(m[["file", "run_id", "rpm", "i_ma", "v_bus"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    X = m[["i_a", "omega"]].to_numpy()
    yv = m["v_bus"].to_numpy()
    beta, *_ = np.linalg.lstsq(X, yv, rcond=None)
    res = yv - X @ beta
    dof = len(yv) - 2
    s2 = res @ res / dof if dof > 0 else np.nan
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    tq = stats.t.ppf(0.975, dof) if dof > 0 else np.nan
    print(f"\nDC モータの式 V_bus = R·I + Ke·ω（切片なし、{len(m)} RUN）：")
    print(f"  R  = {beta[0]:.2f} Ω（95% 区間 {beta[0] - tq * se[0]:.2f}〜{beta[0] + tq * se[0]:.2f}）")
    print(f"  Ke = {beta[1]:.4f} V·s/rad（出力軸。95% 区間 {beta[1] - tq * se[1]:.4f}〜{beta[1] + tq * se[1]:.4f}）")
    print(f"  残差の標準偏差 {np.sqrt(s2) * 1000:.0f} mV")
    print(f"  I と ω の相関 {np.corrcoef(m['i_a'], m['omega'])[0, 1]:+.2f}（±1 に近いと R と Ke を分けにくい）")
    print("  ※ I と ω にばらつきが要るので、電源電圧（V0・V1）と負荷（L0・L1）を変えた RUN を集める。")
    print("     1 RUN の中の 1 ms ごとの行を独立な点にしない（疑似反復）。")


# ---------------- ステップ応答 ----------------
def first_order(t, w0, w1, tau, t0):
    return np.where(t < t0, w0, w0 + (w1 - w0) * (1 - np.exp(-np.clip(t - t0, 0, None) / tau)))


def kalman_loglik(params, y, u, return_states=False):
    """状態 x = [θ（カウント）, ω（カウント/ms）]。1 ms ごと。
    ω[k+1] = a·ω[k] + (1 − a)·(Kc·u[k] + c) + w、θ[k+1] = θ[k] + ω[k]、y[k] = θ[k] + v"""
    log_tau, Kc, c, log_q, log_r = params
    a = np.exp(-1.0 / np.exp(log_tau))
    q, r = np.exp(log_q), 1.0 / 12 + np.exp(log_r)       # 1/12 はカウントの量子化の分散
    th, om = y[0], (y[10] - y[0]) / 10                    # 最初の状態
    P00, P01, P11 = 1.0, 0.0, 1.0
    ll = 0.0
    n = len(y)
    xs = np.zeros((n, 2))
    Ps = np.zeros((n, 3))
    xp_s = np.zeros((n, 2))
    Pp_s = np.zeros((n, 3))
    for k in range(n):
        # 観測で更新（y が欠けていれば予測だけ）
        xp_s[k] = th, om
        Pp_s[k] = P00, P01, P11
        if not np.isnan(y[k]):
            S = P00 + r
            e = y[k] - th
            K0, K1 = P00 / S, P01 / S
            th, om = th + K0 * e, om + K1 * e
            P00, P01, P11 = P00 - K0 * P00, P01 - K0 * P01, P11 - K1 * P01
            ll += -0.5 * (np.log(2 * np.pi * S) + e * e / S)
        xs[k] = th, om
        Ps[k] = P00, P01, P11
        # 予測：F = [[1, 1], [0, a]]
        th, om = th + om, a * om + (1 - a) * (Kc * u[k] + c)
        P00, P01, P11 = P00 + 2 * P01 + P11, a * (P01 + P11), a * a * P11 + q
    if return_states:
        return ll, xs, Ps, xp_s, Pp_s, a
    return ll


def rts_omega(params, y, u):
    """RTS 平滑化で ω を求める（後ろ向きに戻って、あとの観測も使う）"""
    ll, xs, Ps, xp, Pp, a = kalman_loglik(params, y, u, return_states=True)
    n = len(y)
    F = np.array([[1, 1], [0, a]])
    xsm = xs.copy()
    for k in range(n - 2, -1, -1):
        Pf = np.array([[Ps[k, 0], Ps[k, 1]], [Ps[k, 1], Ps[k, 2]]])
        Pn = np.array([[Pp[k + 1, 0], Pp[k + 1, 1]], [Pp[k + 1, 1], Pp[k + 1, 2]]])
        J = Pf @ F.T @ np.linalg.inv(Pn)
        xsm[k] = xs[k] + J @ (xsm[k + 1] - xp[k + 1])
    return xsm[:, 1]


def num_hessian(f, x, h=1e-3):
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            e_i, e_j = np.eye(n)[i] * h, np.eye(n)[j] * h
            H[i, j] = H[j, i] = (f(x + e_i + e_j) - f(x + e_i - e_j) - f(x - e_i + e_j) + f(x - e_i - e_j)) / (4 * h * h)
    return H


def cmd_step(a):
    df = load(a.csv)
    if "run_id" not in df:
        df["run_id"] = 0
    runs = [a.run] if a.run is not None else sorted(df["run_id"].unique())
    to_rpm = 60000 / a.cpr_out
    res = []
    keep = None
    for rid in runs:
        g = df[df["run_id"] == rid].sort_values("t_ms")
        dd = g["duty"].to_numpy()
        sw = np.flatnonzero(np.diff(dd) > 0)
        sw = [i + 1 for i in sw if dd[i] > 0]          # lo → hi に上がった位置（0 → lo は使わない）
        if not sw:
            print(f"run {rid}：lo → hi の切り替えが見つかりません")
            continue
        i0 = sw[0]
        t0 = g["t_ms"].iloc[i0]
        lo, hi = dd[i0 - 1], dd[i0]
        seg = g[(g["t_ms"] >= t0 - 300) & (g["t_ms"] < t0 + 600)]
        full = pd.DataFrame({"t_ms": np.arange(t0 - 300, t0 + 600)}).merge(seg, on="t_ms", how="left")
        if full["count"].isna().any():
            print(f"run {rid}：{int(full['count'].isna().sum())} ms 分の行が欠けている（予測だけでつなぐ）")
        full["duty"] = np.where(full["t_ms"] < t0, lo, hi)
        tt = full["t_ms"].to_numpy(float) - t0
        y = full["count"].to_numpy(float)
        y = y - y[0]
        u = full["duty"].to_numpy(float)

        # (1) 10 ms の差分から速さ → 非線形最小二乗
        W = 10
        vd = (y[W:] - y[:-W]) / W                        # カウント/ms、時刻は区間の中央
        td = tt[:-W] + W / 2
        ok = ~np.isnan(vd)
        p0 = [vd[td < 0].mean(), vd[td > 400].mean(), 30.0, 0.0]
        popt, pcov = optimize.curve_fit(first_order, td[ok], vd[ok], p0=p0)
        se = np.sqrt(np.diag(pcov))
        K_cf = (popt[1] - popt[0]) / (hi - lo) * to_rpm
        tau_cf = popt[2]

        # (2) 状態空間モデル＋カルマンフィルタ（カウントそのものを観測として最尤法）
        Kc0 = (popt[1] - popt[0]) / (hi - lo)
        x0 = np.array([np.log(max(tau_cf, 5)), Kc0, popt[0] - Kc0 * lo, np.log(1e-3), np.log(0.1)])
        nll = lambda p: -kalman_loglik(p, y, u)
        opt = optimize.minimize(nll, x0, method="Nelder-Mead",
                                options={"maxiter": 4000, "xatol": 1e-4, "fatol": 1e-4})
        opt = optimize.minimize(nll, opt.x, method="Nelder-Mead", options={"maxiter": 2000, "xatol": 1e-5, "fatol": 1e-5})
        H = num_hessian(nll, opt.x)
        try:
            cov = np.linalg.inv(H)
            se_ss = np.sqrt(np.clip(np.diag(cov), 0, None))
        except np.linalg.LinAlgError:
            se_ss = np.full(5, np.nan)
        tau_ss = np.exp(opt.x[0])
        tau_ci = np.exp(opt.x[0] + np.array([-1.96, 1.96]) * se_ss[0])   # 対数で正規近似
        K_ss = opt.x[1] * to_rpm
        K_ci = (opt.x[1] + np.array([-1.96, 1.96]) * se_ss[1]) * to_rpm
        print(f"run {rid}：{lo}→{hi}‰（t0 = {t0} ms）")
        print(f"  差分＋curve_fit   ：τ = {tau_cf:6.1f} ms（±{1.96 * se[2]:.1f}）、K = {K_cf:.4f} rpm/‰")
        print(f"  状態空間＋カルマン：τ = {tau_ss:6.1f} ms（95% {tau_ci[0]:.1f}〜{tau_ci[1]:.1f}）、"
              f"K = {K_ss:.4f} rpm/‰（95% {K_ci[0]:.4f}〜{K_ci[1]:.4f}）、"
              f"q = {np.exp(opt.x[3]):.2e}、r の上乗せ = {np.exp(opt.x[4]):.2e}")
        res.append({"run": rid, "tau_cf": tau_cf, "K_cf": K_cf, "tau_ss": tau_ss, "K_ss": K_ss,
                    "rpm_lo": popt[0] * to_rpm, "rpm_hi": popt[1] * to_rpm})
        if keep is None:
            keep = (rid, tt, td, vd, popt, opt.x, y, u)

    if not res:
        sys.exit("解析できるステップがありませんでした")
    r = pd.DataFrame(res)
    print(f"\n{len(r)} 回のステップ（独立な単位＝ステップ1回）の平均 ± 標準偏差：")
    for c, unit in [("tau_cf", "ms"), ("tau_ss", "ms"), ("K_cf", "rpm/‰"), ("K_ss", "rpm/‰")]:
        print(f"  {c:7s} = {r[c].mean():.4g} ± {r[c].std(ddof=1) if len(r) > 1 else float('nan'):.2g} {unit}")
    if len(r) > 1:
        tq = stats.t.ppf(0.975, len(r) - 1)
        m, s = r["tau_ss"].mean(), r["tau_ss"].std(ddof=1)
        print(f"  τ（状態空間）の 95% 区間（ステップ間のばらつきから）：{m - tq * s / np.sqrt(len(r)):.1f}〜{m + tq * s / np.sqrt(len(r)):.1f} ms")
    print("※ 差分は 10 ms の平均なので、速い変化がなまり、τ が少し大きめに出ることがある。")

    if a.plot and keep is not None:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
        rid, tt, td, vd, popt, px, y, u = keep
        om = rts_omega(px, y, u)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(td, vd * to_rpm, ".", ms=3, color="0.6", label="10 ms の差分")
        ax.plot(tt, om * to_rpm, "b-", label="カルマン（RTS 平滑化）")
        ax.plot(td, first_order(td, *popt) * to_rpm, "r--", label=f"1次遅れ τ = {popt[2]:.1f} ms")
        ax.set_xlabel("切り替えからの時間 [ms]")
        ax.set_ylabel("出力軸の回転数 [rpm]")
        ax.set_title(f"run {rid} のステップ応答")
        ax.legend()
        fig.tight_layout()
        fig.savefig(a.plot, dpi=120)
        print(f"{a.plot} を保存しました")


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("sweep")
    p.add_argument("csv")
    p.add_argument("--cpr-out", type=float, required=True)
    p.add_argument("--gear", type=float, default=None)
    p.add_argument("--table", default=None)
    p.add_argument("--plot", default="sweep.png")
    p = sub.add_parser("dcfit")
    p.add_argument("csv", nargs="+")
    p.add_argument("--cpr-out", type=float, required=True)
    p = sub.add_parser("step")
    p.add_argument("csv")
    p.add_argument("--cpr-out", type=float, required=True)
    p.add_argument("--run", type=int, default=None)
    p.add_argument("--plot", default="step.png")
    a = ap.parse_args()
    {"sweep": cmd_sweep, "dcfit": cmd_dcfit, "step": cmd_step}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pole_period.py
#!/usr/bin/env python3
"""第4章：ロジックアナライザの記録から、エンコーダの「モータ軸1回転あたりのパルス数 P」を求める

考え方：エンコーダの磁石の極の間隔は、製造のばらつきで少しずつ違う。そのため、一定の速さで回しても
A 相の周期（立ち上がりから次の立ち上がりまで）は、モータ軸1回転ごとに同じ模様をくり返す。
周期の並びの自己相関が最も強くなるずれ（ラグ）が、1回転あたりの周期の数＝P になる。
P が分かれば、手で回して求めた CPR_out（出力軸1回転あたりのカウント）から、減速比 G = CPR_out /(4P)。

使い方: cd ~/home-lab/ch4 && uv run python pole_period.py data/enc_la.bin 2000000 --cpr-out 2744 --plot pole.png
  BIN : sigrok-cli -O binary の出力（1サンプル1バイト、ビット k がチャネル Dk）
  FS  : サンプリング周波数 [Hz]
"""
import argparse
import sys

import numpy as np


def rising_edges(bits):
    return np.flatnonzero((bits[1:] == 1) & (bits[:-1] == 0)) + 1


def moving_median(x, w):
    h = w // 2
    xp = np.pad(x, h, mode="edge")
    win = np.lib.stride_tricks.sliding_window_view(xp, w)
    return np.median(win, axis=1)


def acf(x, max_lag):
    x = x - x.mean()
    d = np.dot(x, x)
    return np.array([np.dot(x[:-k], x[k:]) / d for k in range(1, max_lag + 1)])


def pick_period(r, ratio=0.7):
    """最大のピークのラグ L を探し、その約数のラグにも同じくらい強いピークがあれば小さいほうを選ぶ"""
    lags = np.arange(1, len(r) + 1)
    L = int(lags[np.argmax(r)])
    for k in range(L // 2, 0, -1):
        if L % k == 0 and k >= 2 and r[k - 1] >= ratio * r[L - 1]:
            L = k
    return L


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("bin")
    ap.add_argument("fs", type=float)
    ap.add_argument("--ch-a", type=int, default=0)
    ap.add_argument("--ch-b", type=int, default=1)
    ap.add_argument("--max-lag", type=int, default=60)
    ap.add_argument("--cpr-out", type=float, default=None, help="出力軸1回転あたりのカウント（手で回して求めた値）")
    ap.add_argument("--n-shuffle", type=int, default=2000)
    ap.add_argument("--plot", default=None)
    a = ap.parse_args()

    raw = np.fromfile(a.bin, dtype=np.uint8)
    A = (raw >> a.ch_a) & 1
    B = (raw >> a.ch_b) & 1
    print(f"{len(raw)} サンプル（{len(raw) / a.fs:.3f} 秒）")
    ea = rising_edges(A)
    if len(ea) < 5 * a.max_lag:
        sys.exit(f"A 相の立ち上がりが {len(ea)} 個しかありません。モータが回っているか、--ch-a が正しいかを確かめてください")
    per = np.diff(ea) / a.fs                          # A 相の周期 [s]
    # 向き：A の立ち上がりのときの B の値（0 と 1 のどちらが多いか）
    b_at = B[ea]
    print(f"A 相の周期の数 {len(per)}、平均 {per.mean() * 1e6:.1f} µs（{1 / per.mean():.1f} Hz）、"
          f"A の立ち上がりで B = 1 の割合 {b_at.mean():.2f}（0 か 1 に近ければ A・B の位相差は正常）")
    q = 1 / a.fs / per.mean()
    print(f"時間の分解能（1サンプル）は周期の {q * 100:.2f}%")
    dev = per / moving_median(per, 51) - 1           # ゆっくりした速さの変化を割って除く
    print(f"周期のばらつき（ゆっくりした変化を除いたもの）：標準偏差 {dev.std() * 100:.2f}%")
    r = acf(dev, a.max_lag)
    L = pick_period(r)
    # 並べ替え検定：周期の順番をでたらめにしたとき、どのラグでも r(L) 以上の自己相関が出る確率
    rng = np.random.default_rng(0)
    mx = np.empty(a.n_shuffle)
    for i in range(a.n_shuffle):
        mx[i] = acf(rng.permutation(dev), a.max_lag).max()
    p = (np.sum(mx >= r[L - 1]) + 1) / (a.n_shuffle + 1)
    print("ラグ 1〜15 の自己相関：" + " ".join(f"{k}:{r[k - 1]:+.2f}" for k in range(1, 16)))
    print(f"\n→ くり返しの周期 L = {L}（r = {r[L - 1]:.3f}、並べ替え検定 p = {p:.4f}、"
          f"倍数のラグ {2 * L}・{3 * L} の r = {r[2 * L - 1]:.2f}・{r[3 * L - 1] if 3 * L <= a.max_lag else float('nan'):.2f}）")
    print(f"   モータ軸1回転あたり、A 相のパルス P = {L}（4逓倍で {4 * L} カウント）")
    if p > 0.01:
        print("   ⚠ p が大きい：ばらつきの模様がはっきりしない。記録を長くする・fs を上げる・速さを一定にする")
    if a.cpr_out:
        G = a.cpr_out / (4 * L)
        print(f"   減速比 G = CPR_out /(4P) = {a.cpr_out:g} / {4 * L} = {G:.2f}")
        print(f"   モータ軸の回転数 = {1 / per.mean() / L * 60:.0f} rpm、出力軸 = {1 / per.mean() / L * 60 / G:.1f} rpm")
    if a.plot:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        n = min(len(dev), 6 * L)
        ax[0].plot(np.arange(n), dev[:n] * 100, "o-")
        for k in range(0, n, L):
            ax[0].axvline(k, color="0.8")
        ax[0].set_xlabel("A 相の周期の番号")
        ax[0].set_ylabel("周期のずれ [%]")
        ax[0].set_title(f"同じ模様が {L} 周期ごとにくり返す")
        ax[1].bar(np.arange(1, a.max_lag + 1), r)
        ax[1].axhline(np.quantile(mx, 0.99), color="r", ls="--", label="並べ替えの最大値の99%点")
        ax[1].set_xlabel("ラグ（周期の数）")
        ax[1].set_ylabel("自己相関")
        ax[1].legend()
        fig.tight_layout()
        fig.savefig(a.plot, dpi=120)
        print(f"{a.plot} を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile split_plot.py
#!/usr/bin/env python3
"""第4章：電源電圧 × 負荷 × デューティの要因計画（分割区法）を作り、分散分析する

電源電圧（V：ダイオードの有無）と負荷（L：糸ブレーキの有無）は、配線や取り付けを変えないと変えられない
（変えにくい因子）。そこで「V と L の組み合わせを1回セットアップ（setup_id）したら、その中でデューティを
3つ、でたらめな順に測る」分割区法にする。V・L の効果を調べるときの独立な単位はセットアップで、
デューティの1回1回ではない。

使い方（~/home-lab/ch4 で）:
  uv run python split_plot.py design --reps 2 --seed 0 --out data/fact/design.csv
      → セットアップの順番と、それぞれで実行する motor_run.py のコマンドを表示する
  uv run python split_plot.py analyze data/fact/design.csv --cpr-out 2744
      → design.csv の file 列の CSV（motor_run.py sweep の出力）を読み、定常の回転数を求めて分散分析
"""
import argparse
import csv
import pathlib
import sys
import warnings

import numpy as np
import pandas as pd

from motor_analysis import summarize_sweep

DUTIES = [400, 700, 1000]


def cmd_design(a):
    out = pathlib.Path(a.out)
    if out.exists():
        sys.exit(f"{out} はすでにあります（上書きしないように止めました）")
    out.parent.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(a.seed)
    combos = [(v, l) for v in ("V0", "V1") for l in ("L0", "L1")] * a.reps
    order = rng.permutation(len(combos))
    rows = []
    for k, i in enumerate(order):
        v, l = combos[i]
        duties = list(rng.permutation(DUTIES))
        sid = f"S{k + 1:02d}"
        rows.append({"setup_id": sid, "order": k + 1, "V": v, "L": l,
                     "duty_order": " ".join(str(d) for d in duties), "file": f"{sid}.csv"})
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    print(f"{out} を保存しました。上から順に、セットアップを作り直して測ります")
    print("  V0 = ダイオードなし、V1 = 1N5819 を2本直列、L0 = 負荷なし、L1 = 糸ブレーキ")
    for r in rows:
        print(f"  {r['setup_id']}：{r['V']} {r['L']} → uv run python motor_run.py sweep "
              f"--duties {r['duty_order'].replace(' ', ',')} --rep 1 --order up --ms 3000 "
              f"--out {out.parent / r['file']}")


def anova_table(df):
    """分割区法の分散分析（釣り合った計画を仮定）。V・L・V:L はセットアップの平均の上で、残りは行の上で計算する"""
    import statsmodels.formula.api as smf
    from statsmodels.stats.anova import anova_lm
    from scipy import stats
    # 誤った分析：すべての行を独立とみなした三元配置
    wrong = anova_lm(smf.ols("rpm ~ C(V) * C(L) * C(duty)", data=df).fit(), typ=2)
    # 1次の単位（セットアップ）：セットアップごとの平均で二元配置。その残差＝セットアップの間のばらつき
    n_sub = df.groupby("setup_id").size().iloc[0]                    # 1セットアップの中の行の数（3）
    sp = df.groupby(["setup_id", "V", "L"], as_index=False)["rpm"].mean()
    wp = anova_lm(smf.ols("rpm ~ C(V) * C(L)", data=sp).fit(), typ=2)
    ms_wp = wp.loc["Residual", "sum_sq"] * n_sub / wp.loc["Residual", "df"]
    df_wp = wp.loc["Residual", "df"]
    # 2次の単位（セットアップの中のデューティ）：デューティを含む項の平方和は三元配置と同じ。
    # 三元配置の残差から「セットアップの間のばらつき」を引いたものが、セットアップの中の残差
    ss_wp = wp.loc["Residual", "sum_sq"] * n_sub
    df_res = wrong.loc["Residual", "df"] - df_wp
    ms_res = (wrong.loc["Residual", "sum_sq"] - ss_wp) / df_res
    sub = wrong
    right = []
    for term in ["C(V)", "C(L)", "C(V):C(L)"]:
        ss, d = wp.loc[term, "sum_sq"] * n_sub, wp.loc[term, "df"]
        F = ss / d / ms_wp
        right.append({"項": term, "分母": "セットアップ", "F": F, "df": f"{d:.0f}, {df_wp:.0f}", "p": stats.f.sf(F, d, df_wp)})
    for term in ["C(duty)", "C(V):C(duty)", "C(L):C(duty)", "C(V):C(L):C(duty)"]:
        ss, d = sub.loc[term, "sum_sq"], sub.loc[term, "df"]
        F = ss / d / ms_res
        right.append({"項": term, "分母": "残差", "F": F, "df": f"{d:.0f}, {df_res:.0f}", "p": stats.f.sf(F, d, df_res)})
    return wrong, pd.DataFrame(right).set_index("項"), ms_wp, df_wp, ms_res, df_res


def cmd_analyze(a):
    design = pd.read_csv(a.design)
    base = pathlib.Path(a.design).parent
    rows = []
    for _, s in design.iterrows():
        path = base / s["file"]
        if not path.exists():
            sys.exit(f"{path} がありません（{s['setup_id']} をまだ測っていない？）")
        t = summarize_sweep(pd.read_csv(path), a.cpr_out)
        for _, r in t.iterrows():
            rows.append({"setup_id": s["setup_id"], "order": s["order"], "V": s["V"], "L": s["L"],
                         "duty": int(r["duty_set"]), "rpm": r["rpm"], "i_ma": r["i_ma"], "v_bus": r["v_bus"]})
    df = pd.DataFrame(rows)
    pd.set_option("display.width", 130)
    print("セル平均（出力軸の rpm）：")
    print(df.pivot_table(index=["V", "L"], columns="duty", values="rpm").round(1).to_string())
    print("\nセットアップごとの電源電圧の平均（INA219、V）：")
    print(df.groupby(["setup_id", "V", "L"])["v_bus"].mean().round(3).to_string())

    wrong, right, ms_wp, df_wp, ms_res, df_res = anova_table(df)
    print("\n(a) 誤った分析：すべての行を独立なくり返しとみなした三元配置の分散分析")
    print(wrong[["df", "F", "PR(>F)"]].round(4).to_string())
    print("\n(b) 分割区法：V・L・V:L はセットアップの平均平方で、デューティを含む項は残差で検定")
    print(f"    セットアップの平均平方 {ms_wp:.2f}（自由度 {df_wp:.0f}）、残差の平均平方 {ms_res:.2f}（自由度 {df_res:.0f}）")
    print(right.round(4).to_string())

    import statsmodels.formula.api as smf
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mm = smf.mixedlm("rpm ~ C(V) * C(L) * C(duty)", df, groups=df["setup_id"]).fit(reml=True)
    print("\n(c) 混合モデル（セットアップを変量効果）の主な係数（基準 V0・L0・デューティ 400 からの差）：")
    for k in ["C(V)[T.V1]", "C(L)[T.L1]", "C(V)[T.V1]:C(L)[T.L1]"]:
        print(f"    {k:24s} {mm.params[k]:8.2f}（標準誤差 {mm.bse[k]:.2f}、p = {mm.pvalues[k]:.4f}）")
    print(f"    セットアップの分散 {float(mm.cov_re.iloc[0, 0]):.2f}、残差の分散 {mm.scale:.2f}")
    print("    ※ この係数は「V0・L0・デューティ 400 のところでの差」で、(b) の主効果（デューティで平均した効果）とは別もの。")
    print("    ※ p 値は正規分布の近似で、区の数が少ない（V・L の自由度 4）ことを考えないので当てにならない（(b) とは比べられない）。")
    print("       V・L の検定には (b) を使う。")
    print("\n※ (a) は V・L の効果を、同じセットアップの中の3回（デューティ違い）まで独立な証拠として数えるので、")
    print("   p 値が小さすぎる（疑似反復）。V・L について独立なのは、セットアップを作り直した回数だけ。")


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("design")
    p.add_argument("--reps", type=int, default=2)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--out", required=True)
    p = sub.add_parser("analyze")
    p.add_argument("design")
    p.add_argument("--cpr-out", type=float, required=True)
    a = ap.parse_args()
    {"design": cmd_design, "analyze": cmd_analyze}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile synth_motor.py
#!/usr/bin/env python3
"""第4章：解析コードを試すための合成データを作る（実機がなくても、解析の流れを最後まで確かめられる）

使い方（~/home-lab/ch4 で）:
  uv run python synth_motor.py sweep --out data_synth/sweep.csv
  uv run python synth_motor.py step  --out data_synth/step.csv
  uv run python synth_motor.py la    --out data_synth/la.bin       # sigrok-cli -O binary と同じ形（2MHz）
  uv run python synth_motor.py fact  --design data_synth/design.csv # 設計表の file 列に、セットアップごとの CSV を作る
真の値：P = 7（モータ軸1回転あたりのパルス）、G = 98（減速比）→ CPR_out = 4·7·98 = 2744、
       τ = 40 ms、R = 8 Ω、Ke = 0.161 V·s/rad（出力軸）、摩擦の電流 60 mA、
       コイルの L/R = 50 µs、D1 の順電圧 0.3 V。PWM のオフの間の電流が途切れる（不連続導通）ことも計算する。
       i_ma は INA219 と同じく電池側の電流（PWM のオンの間だけ流れる電流の平均）。
  sweep には --duties 500 --rep 3 --freq 4000 のように、デューティ・くり返し・PWM の周波数を指定できる
"""
import argparse
import csv
import json
import pathlib
import sys

import numpy as np

P_TRUE, G_TRUE = 7, 98
CPR_OUT = 4 * P_TRUE * G_TRUE
TAU_MS, R_OHM, KE, I_F = 40.0, 8.0, 0.161, 0.060
FIELDS = ["t_ms", "duty", "count", "ina_seq", "i_ma", "v_bus"]


TAU_E, V_D = 50e-6, 0.3                            # コイルの L/R [s]、D1 の順電圧 [V]


def _seg(i0, a, t):
    """i(t) = a + (i0 - a)·exp(-t/τe) を、0 より下がらないとして長さ t だけ進める。(終わりの電流, 積分)"""
    if i0 <= 0 and a <= 0:
        return 0.0, 0.0
    tz = t
    if a < 0:                                       # 0 に届く時刻
        tz = min(t, TAU_E * np.log((i0 - a) / (-a)))
    e = np.exp(-tz / TAU_E)
    integ = a * tz + (i0 - a) * TAU_E * (1 - e)
    i_end = a + (i0 - a) * e if tz == t else 0.0
    return max(i_end, 0.0), integ


def pwm_currents(e_back, d, f, v):
    """逆起電力 e_back のとき、PWM（デューティ d、周波数 f）の1周期の平均の電流（モータ、電池側）[A]。
    オフの間はコイルの電流が D1 を回り、0 になったらそこで止まる（不連続導通）"""
    if d >= 1:
        i = max((v - e_back) / R_OHM, 0.0)
        return i, i
    if d <= 0:
        return 0.0, 0.0
    ton, toff = d / f, (1 - d) / f
    i0 = 0.0
    for _ in range(60):                             # 定常の周期解になるまでくり返す
        i1, q_on = _seg(i0, (v - e_back) / R_OHM, ton)
        i2, q_off = _seg(i1, (-V_D - e_back) / R_OHM, toff)
        if abs(i2 - i0) < 1e-9:
            break
        i0 = i2
    return (q_on + q_off) * f, q_on * f


_cache = {}


def steady_e(d, f, v, i_need):
    """平均のモータ電流が i_need（摩擦＋負荷）になる逆起電力 [V]。回らなければ 0"""
    key = (round(d, 4), f, round(v, 3), round(i_need, 5))
    if key not in _cache:
        if pwm_currents(0.0, d, f, v)[0] <= i_need:
            _cache[key] = 0.0
        else:
            lo, hi = 0.0, v
            for _ in range(40):                     # 二分法
                mid = (lo + hi) / 2
                if pwm_currents(mid, d, f, v)[0] > i_need:
                    lo = mid
                else:
                    hi = mid
            _cache[key] = lo
    return _cache[key]


def simulate(sched, rng, v_batt=5.2, load_a=0.0, state=None, freq=1000.0):
    """sched：[(duty, ms), ...]。最後に duty 0 で 300 ms。1 ms ごとの D 行を返す。
    i_ma は INA219 と同じく電池側の電流（Q1 の上流。オフの間に D1 を回る電流は入らない）"""
    sched = list(sched) + [(0, 300)]
    duty = np.concatenate([np.full(ms, d) for d, ms in sched])
    n = len(duty)
    theta, w = (state or (0.0, 0.0))
    a = np.exp(-1.0 / TAU_MS)
    r_src = 0.5                                     # 電池・ヒューズ・ポリスイッチ・配線などの抵抗 [Ω]
    rows, seq, i_last, v_last = [], 0, 0.0, v_batt
    rad_per_count = 2 * np.pi / CPR_OUT
    i_acc = []
    v_bus = v_batt
    for k in range(n):
        d = duty[k] / 1000
        w_rad = w * rad_per_count * 1000              # カウント/ms → rad/s
        if d > 0:
            i_bat = pwm_currents(KE * w_rad, d, freq, v_bus)[1]
            v_bus = v_batt - r_src * i_bat
            e_tgt = steady_e(d, freq, round(v_bus, 2), I_F + load_a)
            w_tgt = e_tgt / KE                        # rad/s
        else:
            v_bus, i_bat, w_tgt = v_batt, 0.0, 0.0
        w_tgt_c = w_tgt / rad_per_count / 1000
        w = a * w + (1 - a) * w_tgt_c + rng.normal(0, 0.02)
        w = max(w, 0.0)
        theta += w
        i_acc.append(i_bat)
        if k % 9 == 8:                                 # INA219 は約9 ms ごとに新しい値
            seq += 1
            i_last = np.mean(i_acc[-8:]) * 1000 + rng.normal(0, 1.0)
            v_last = v_bus + rng.normal(0, 0.004)
        rows.append({"t_ms": k, "duty": int(duty[k]), "count": int(np.floor(theta)),
                     "ina_seq": seq, "i_ma": f"{i_last:.1f}", "v_bus": f"{v_last:.3f}"})
    return rows, (theta, w)


def write(out, rows, cols):
    out = pathlib.Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        wr.writeheader()
        wr.writerows(rows)
    print(f"{out}（{len(rows)} 行）")


def sweep_rows(duties, ms, rng, v_batt=5.2, load_a=0.0, rep_of=None, freq=1000.0):
    rows, st = [], (0.0, 0.0)
    for k, d in enumerate(duties):
        r, st = simulate([(d, ms)], rng, v_batt, load_a, st, freq)
        for x in r:
            x.update(run_id=k, duty_set=d, rep=(rep_of[k] if rep_of else 0), host_time="0")
        rows += r
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("kind", choices=["sweep", "step", "la", "fact"])
    ap.add_argument("--out", default=None)
    ap.add_argument("--design", default=None)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--duties", default=",".join(str(d) for d in range(0, 1001, 100)), help="sweep のデューティ（千分率）")
    ap.add_argument("--rep", type=int, default=2, help="sweep のくり返し")
    ap.add_argument("--freq", type=float, default=1000.0, help="PWM の周波数 [Hz]")
    a = ap.parse_args()
    rng = np.random.default_rng(a.seed)
    cols_sweep = ["run_id", "duty_set", "rep"] + FIELDS + ["host_time"]
    if a.kind == "sweep":
        duties = [int(x) for x in a.duties.split(",")]
        plan = []
        for rep in range(a.rep):
            ds = duties[:]
            rng.shuffle(ds)
            plan += [(d, rep) for d in ds]
        rows = sweep_rows([d for d, _ in plan], 3000, rng, rep_of=[r for _, r in plan], freq=a.freq)
        write(a.out, rows, cols_sweep)
    elif a.kind == "step":
        rows, st = [], (0.0, 0.0)
        for k in range(5):
            r, st = simulate([(300, 1500), (700, 1500)], rng, state=st, freq=a.freq)
            for x in r:
                x.update(run_id=k, lo=300, hi=700, ms=1500, host_time="0")
            rows += r
        write(a.out, rows, ["run_id", "lo", "hi", "ms"] + FIELDS + ["host_time"])
    elif a.kind == "la":
        fs = 2_000_000
        dur = 1.0
        f_motor = 150 / 60 * G_TRUE                    # 出力軸 150 rpm → モータ軸 [rev/s]
        # 磁石の極の位置（1回転に A の立ち上がり・立ち下がりが 2P 個）。製造のばらつき 3%
        base = np.arange(2 * P_TRUE) / (2 * P_TRUE)
        pos = base + rng.normal(0, 0.03 / (2 * P_TRUE), 2 * P_TRUE)
        pos[0] = 0.0
        n_rev = int(dur * f_motor) + 2
        # 回転角（回転数）＝ f·t ＋ ゆっくりした速さのゆらぎ
        t = np.arange(int(fs * dur)) / fs
        rev = f_motor * t + 0.3 * np.sin(2 * np.pi * 3 * t)
        edges_a = np.sort((np.arange(n_rev)[:, None] + pos[None, :]).ravel())
        edges_b = np.sort((np.arange(n_rev)[:, None] + pos[None, :] + 0.25 / (2 * P_TRUE)).ravel())
        a_bit = (np.searchsorted(edges_a, rev, side="right") % 2).astype(np.uint8)
        b_bit = (np.searchsorted(edges_b, rev, side="right") % 2).astype(np.uint8)
        data = a_bit | (b_bit << 1)
        out = pathlib.Path(a.out)
        out.parent.mkdir(parents=True, exist_ok=True)
        data.tofile(out)
        print(f"{out}（{len(data)} サンプル、fs = {fs} Hz、モータ軸 {f_motor:.1f} rev/s）")
    elif a.kind == "fact":
        if not a.design:
            sys.exit("--design が必要です")
        with open(a.design) as f:
            d = list(csv.DictReader(f))
        base = pathlib.Path(a.design).parent
        for row in d:
            v = 5.2 - (0.6 if row["V"] == "V1" else 0.0) + rng.normal(0, 0.08)   # セットアップごとのずれ
            load = (0.040 if row["L"] == "L1" else 0.0) + max(rng.normal(0, 0.008), -0.02)
            duties = [int(x) for x in row["duty_order"].split()]
            rows = sweep_rows(duties, 3000, rng, v_batt=v, load_a=load)
            write(base / row["file"], rows, cols_sweep)
    print(json.dumps({"P": P_TRUE, "G": G_TRUE, "CPR_out": CPR_OUT, "tau_ms": TAU_MS,
                      "R": R_OHM, "Ke": KE}, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
import glob, json, os, re, shlex, subprocess, sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

CPR = 2744          # 合成データの出力軸1回転のカウント（4·P·G = 4·7·98）。実機では手順6で求めた値
os.makedirs("data_synth", exist_ok=True)


def run(args, check=True):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    words = []
    for w in shlex.split(args):   # data_synth/fact/S*.csv のような * を、ターミナルと同じく名前の順に広げる
        words += (sorted(glob.glob(w)) or [w]) if any(c in w for c in "*?[") else [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True)
    print(p.stdout, end="")
    keep = [l for l in p.stderr.splitlines() if "findfont" not in l and l.strip()]   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
    if keep:
        print("\n".join(keep), file=sys.stderr)
    if check:
        assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=800):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


print("準備ができました")

## 1. デューティのスイープ（手順 7〜9、4-3-8）

本文の `setup_ch4.sh` の最後の2行と同じです。`synth_motor.py sweep` は、デューティ 0〜100%（0〜1000‰ を 100‰ 刻み）を2回ずつ、でたらめな順に 3 秒ずつ回した記録を作ります（数秒）。`motor_analysis.py sweep` は、RUN ごとに最初の 1 秒を捨て、カウントの傾きから回転数を求めます。

In [ ]:
run("synth_motor.py sweep --out data_synth/sweep.csv")
out_sw = run(f"motor_analysis.py sweep data_synth/sweep.csv --cpr-out {CPR} --plot data_synth/sweep.png")
show("data_synth/sweep.png")

In [ ]:
# 確かめ（4-7「スイープ」：合成データでは u0 は 100‰ と 200‰ の間、全速との比は 20% で 49%、40% で 82%、70% で 95%、電流は約 60 mA）
ok("動き出すデューティ u0：100‰ と 200‰ の間" in out_sw, "動き出すデューティ u0 は 100‰ と 200‰ の間")
ratio = {int(m.group(1)): float(m.group(2)) for m in re.finditer(r"(\d+)‰：.*?全速の\s+([\d.]+)%", out_sw)}
for d, want in [(200, 49), (400, 82), (700, 95)]:
    ok(abs(ratio[d] - want) < 1.0, f"デューティ {d/10:.0f}% の回転数は全速の {ratio[d]}%（本文 {want}%。比例なら {d/10:.0f}%）")
t = pd.read_csv("data_synth/sweep.csv")
full = num(r"1000‰：\s+([\d.]+) rpm", out_sw)
ok(200 <= full <= 280, f"全速 {full} rpm（本文の目安 200〜280 rpm）")
sys.path.insert(0, ".")
from motor_analysis import summarize_sweep
tab = summarize_sweep(t, CPR)
moving = tab[tab["rpm"] > 5]
ok(abs(moving["i_ma"].mean() - 60) < 5 and moving["i_ma"].std() < 5,
   f"回っている RUN の電池側の電流は平均 {moving['i_ma'].mean():.1f} mA（約 60 mA）で、デューティにほとんどよらない（sd {moving['i_ma'].std():.1f} mA）")

## 2. ステップ応答（手順 11）

デューティを 30% → 70% に上げるステップを5回くり返した記録です。1回ごとに、(1) 10 ms の差分の速さに1次遅れを curve_fit で当てはめる方法と、(2) カウントそのものを観測とする状態空間モデル＋カルマンフィルタの最尤法で、時定数 τ とゲイン K を求めます（10 秒ほど）。

In [ ]:
run("synth_motor.py step --out data_synth/step.csv")
out_st = run(f"motor_analysis.py step data_synth/step.csv --cpr-out {CPR} --plot data_synth/step.png")
show("data_synth/step.png")

In [ ]:
# 確かめ（4-7「ステップ応答」：差分＋curve_fit 43.4±3.5 ms、状態空間＋カルマン 38.7±2.7 ms（95% 35.3〜42.1）、K 約 0.165 rpm/‰）
tau_cf, sd_cf = num(r"tau_cf  = ([\d.]+) ± ([\d.]+)", out_st), num(r"tau_cf  = [\d.]+ ± ([\d.]+)", out_st)
tau_ss, sd_ss = num(r"tau_ss  = ([\d.]+) ±", out_st), num(r"tau_ss  = [\d.]+ ± ([\d.]+)", out_st)
lo, hi = num(r"95% 区間（ステップ間のばらつきから）：([\d.]+)〜", out_st), num(r"95% 区間（ステップ間のばらつきから）：[\d.]+〜([\d.]+)", out_st)
ok(abs(tau_cf - 43.4) < 0.2 and abs(sd_cf - 3.5) < 0.2, f"差分＋curve_fit：τ = {tau_cf} ± {sd_cf} ms（本文 43.4±3.5）")
ok(abs(tau_ss - 38.7) < 0.2 and abs(sd_ss - 2.7) < 0.2, f"状態空間＋カルマン：τ = {tau_ss} ± {sd_ss} ms（本文 38.7±2.7）")
ok(lo < 40 < hi, f"状態空間の 95% 区間 {lo}〜{hi} ms に真の値 40 ms が入る（本文 35.3〜42.1）")
ok(tau_cf > tau_ss, "差分（10 ms の平均でなまる）のほうが τ が大きめに出る")
K_ss = num(r"K_ss    = ([\d.]+)", out_st)
ok(abs(K_ss - 0.165) < 0.003, f"K = {K_ss} rpm/‰（本文 約 0.165。30% と 70% の間の平均の傾き）")
# 統計の誤用1：curve_fit が1回ごとに出す区間（±）は、5回のばらつきよりずっと狭い
pm = [float(x) for x in re.findall(r"差分＋curve_fit   ：τ =\s+[\d.]+ ms（±([\d.]+)）", out_st)]
ok(len(pm) == 5 and max(pm) < sd_cf, f"curve_fit の1回ごとの区間 ±{min(pm)}〜{max(pm)} ms は、5回の標準偏差 {sd_cf} ms より狭い（統計の誤用1）")

## 3. PWM の周波数（手順 12）

デューティ 50% で、PWM の周波数を 250 Hz〜4 kHz に変えて 3 秒×3回ずつ回した記録です。本文の `run_ch4.sh` の E の2つの `for` を、Python のくり返しにしただけです（数秒）。不連続導通が効くと、周波数が高いほど遅くなります（オフのあと電流が D1 を回って 0 に減るまでの十数µs が周期に占める割合が増え、逆起電力がそのまま現れる区間が短くなるため。4-3-8）。合成データにはゲートの遅れを入れていないので、実機で逆向きになることがある（4-7）点に注意してください。

In [ ]:
freqs = [250, 500, 1000, 2000, 4000]
rpm_f, se_f = {}, {}
for f in freqs:
    run(f"synth_motor.py sweep --duties 500 --rep 3 --freq {f} --out data_synth/freq_{f}.csv")
for f in freqs:
    print(f"===== {f} Hz =====")
    o = run(f"motor_analysis.py sweep data_synth/freq_{f}.csv --cpr-out {CPR} --plot data_synth/freq_{f}.png")
    rpm_f[f] = num(r"回転数の平均 ([\d.]+) rpm", o)
    se_f[f] = num(r"標準誤差 ([\d.]+)", o)
print({f: rpm_f[f] for f in freqs})

In [ ]:
# 確かめ（4-7「PWM の周波数」：250 Hz で 248.1、1 kHz で 243.8、4 kHz で 222.8 rpm、標準誤差 0.24、約10% 下がる）
for f, want in [(250, 248.1), (1000, 243.8), (4000, 222.8)]:
    ok(abs(rpm_f[f] - want) < 0.2, f"{f} Hz：{rpm_f[f]} rpm（本文 {want}）")
ok(all(abs(se_f[f] - 0.24) < 0.05 for f in freqs), "どの周波数でも標準誤差は約 0.24 rpm")
ok(all(rpm_f[a] > rpm_f[b] for a, b in zip(freqs, freqs[1:])), "周波数が高いほど遅い（不連続導通）")
drop = 1 - rpm_f[4000] / rpm_f[250]
ok(0.08 < drop < 0.12 and (rpm_f[250] - rpm_f[4000]) > 10 * se_f[250],
   f"250 Hz → 4 kHz で {drop*100:.1f}% 下がり、差は標準誤差の何十倍もある")

## 4. P と G（手順 13）

本文では、Pi の `sigrok-cli -O binary` でエンコーダの A・B 相を記録します（1サンプル1バイト、ビット 0 が A、ビット 1 が B）。`synth_motor.py la` は、同じ形の 1 秒の記録（2 MHz。本文の手順は 4 MHz）を作ります。磁石の極の間隔に 3% のばらつきを入れてあるので、A 相の周期の並びは 7 周期ごとに同じ模様をくり返します。`pole_period.py` の2番めの引数は、サンプリング周波数（ここでは 2000000）です。

In [ ]:
run("synth_motor.py la --out data_synth/la.bin")
out_la = run(f"pole_period.py data_synth/la.bin 2000000 --cpr-out {CPR} --plot data_synth/pole.png")
show("data_synth/pole.png")

In [ ]:
# 確かめ（4-7「P と G」：合成データでは L = 7、r = 0.989、p = 0.0005、G = 2744/28 = 98.00）
L = int(num(r"くり返しの周期 L = (\d+)", out_la))
r = num(r"くり返しの周期 L = \d+（r = ([\d.]+)", out_la)
p = num(r"並べ替え検定 p = ([\d.]+)", out_la)
G = num(r"減速比 G = .*? = ([\d.]+)\n", out_la)
ok(L == 7, f"くり返しの周期 L = {L}（真の値 P = 7）")
ok(abs(r - 0.989) < 0.002 and r >= 0.9, f"ラグ L の自己相関 r = {r}（本文 0.989。0.9 以上）")
ok(p <= 0.0005, f"並べ替え検定 p = {p}（2000 回の並べ替えで取りうる最小の値 1/2001 ≈ 0.0005。表示は小数4桁）")
ok(abs(G - 98.00) < 0.005, f"減速比 G = {G}（真の値 98）")
ok(num(r"出力軸 = ([\d.]+) rpm", out_la) == 150.0, "出力軸 150.0 rpm（合成データで決めた速さ）")

## 5. 分割区法の要因計画（手順 14）

電源電圧 V（V0：ダイオードなし、V1：1N5819 を2本直列）と負荷 L（L0：なし、L1：糸ブレーキ）は、配線や取り付けを変えないと変えられない因子です。そこで、V と L の組み合わせを1回セットアップしたら、その中でデューティ 40・70・100% をでたらめな順に測ります（分割区法）。`split_plot.py design` が8つのセットアップの順番を決め、実機ならその表示どおりに `motor_run.py sweep` を8回実行します。ここでは代わりに `synth_motor.py fact` が、セットアップごとの電源電圧と負荷のずれ（区ごとのずれ）を入れて、8つの CSV を作ります。

本文の例は `design --reps 2 --seed 0` と `fact --seed 0` です（本文の手順 G のコマンドは `--seed 7` と書き方の例が違うだけで、やることは同じ）。`design` は、同じ名前の design.csv がすでにあると上書きしないように止まるので、最初に消しておきます。

In [ ]:
import shutil
shutil.rmtree("data_synth/fact", ignore_errors=True)       # もう一度実行するとき用（design は上書きしないで止まるため）
run("split_plot.py design --reps 2 --seed 0 --out data_synth/fact/design.csv")
run("synth_motor.py fact --seed 0 --design data_synth/fact/design.csv")
out_sp = run(f"split_plot.py analyze data_synth/fact/design.csv --cpr-out {CPR}")

In [ ]:
# 確かめ（4-7「分割区法」：V の F 値 (a) 77.5、(b) 30.7（自由度 1, 4）、V×L の p 値 (a) 0.073、(b) 0.28）
design = pd.read_csv("data_synth/fact/design.csv")
ok(len(design) == 8 and design.groupby(["V", "L"]).size().eq(2).all(), "設計表は 8 セットアップ（V×L の4通りを2回ずつ）")
ok(all(sorted(map(int, s.split())) == [400, 700, 1000] for s in design["duty_order"]), "各セットアップでデューティ 400・700・1000 を1回ずつ")
a_part = out_sp.split("(a)")[1].split("(b)")[0]
b_part = out_sp.split("(b) 分割区法")[1].split("(c)")[0]
F_a = num(r"C\(V\)\s+1\.0\s+([\d.]+)", a_part)
p_a_VL = num(r"C\(V\):C\(L\)\s+1\.0\s+[\d.]+\s+([\d.]+)", a_part)
F_b = num(r"C\(V\)\s+セットアップ\s+([\d.]+)\s+1, 4", b_part)
p_b_VL = num(r"C\(V\):C\(L\)\s+セットアップ\s+[\d.]+\s+1, 4\s+([\d.]+)", b_part)
ok(abs(F_a - 77.5) < 0.1 and abs(F_b - 30.7) < 0.1, f"V の F 値：(a) 誤った分析 {F_a}、(b) 分割区法 {F_b}（本文 77.5・30.7）")
ok(abs(p_a_VL - 0.073) < 0.001 and abs(p_b_VL - 0.28) < 0.01,
   f"V×L の p 値：(a) {p_a_VL}、(b) {p_b_VL}（本文 0.073・0.28。(a) のほうが {p_b_VL/p_a_VL:.1f} 倍小さい）")
p_b_D = num(r"C\(duty\)\s+残差\s+[\d.]+\s+2, 8\s+([\d.]+)", b_part)
ok(p_b_D < 1e-3, "D（デューティ）の効果は分割区法でもはっきり出る")

合成データには V×L の交互作用の項を入れていません（V と L は回転数に別々に効くように作った）。それでも (a) の p 値は (b) より 4 倍ほど小さく出ます。(a) は、同じセットアップの中の3回（デューティ違い）まで独立なくり返しとして数えるからです（疑似反復）。本文のとおり、`synth_motor.py fact` の `--seed` を変えると、(a) だけが p < 0.05 になる（偽陽性）ことがあります。次のセルで、seed を 1〜10 に変えて数えてみます（約20秒）。

In [ ]:
# 発展：seed を変えて、(a) と (b) で V×L が p < 0.05 になる回数を数える（V×L の真の効果は 0）
cnt_a = cnt_b = 0
N_SEED = 10            # 増やすと時間がかかる（1回 約2秒）
for s in range(1, N_SEED + 1):
    for f in glob.glob("data_synth/fact/S*.csv"):
        os.remove(f)
    subprocess.run([sys.executable, "synth_motor.py", "fact", "--seed", str(s), "--design", "data_synth/fact/design.csv"],
                   capture_output=True, check=True)
    o = subprocess.run([sys.executable, "split_plot.py", "analyze", "data_synth/fact/design.csv", "--cpr-out", str(CPR)],
                       capture_output=True, text=True, check=True).stdout
    pa = num(r"C\(V\):C\(L\)\s+1\.0\s+[\d.]+\s+([\d.]+)", o.split("(a)")[1].split("(b)")[0])
    pb = num(r"C\(V\):C\(L\)\s+セットアップ\s+[\d.]+\s+1, 4\s+([\d.]+)", o.split("(b) 分割区法")[1])
    cnt_a += pa < 0.05
    cnt_b += pb < 0.05
    print(f"seed {s:2d}：V×L の p 値 (a) {pa:.4f}  (b) {pb:.4f}")
print(f"p < 0.05 になった回数：(a) {cnt_a}/{N_SEED}、(b) {cnt_b}/{N_SEED}")
ok(cnt_a >= cnt_b, "(a) 誤った分析のほうが、真の効果のない V×L を「有意」にしやすい")
# 本文の例（seed 0）のデータに戻す
for f in glob.glob("data_synth/fact/S*.csv"):
    os.remove(f)
run("synth_motor.py fact --seed 0 --design data_synth/fact/design.csv");

## 6. R と K_e（dcfit）と、全部の RUN で当てはめる誤り（手順 14、4-7）

`motor_analysis.py dcfit` は、デューティ 100%（PWM なし）の RUN だけを使って、DC モータの式 $V_{\mathrm{bus}} = R\,I + K_e\,\omega$ を当てはめます。100% なら INA219 の電池側の電流＝モータの電流で、モータにかかる電圧＝$V_{\mathrm{bus}}$ だからです。電源電圧と負荷を変えた 5 節の8つのセットアップの 100% の RUN（1 RUN = 1 点、8 点）を集めます。

In [ ]:
out_dc = run(f"motor_analysis.py dcfit data_synth/fact/S*.csv --cpr-out {CPR}")

In [ ]:
# 確かめ（4-7「R と K_e」：R = 8.11 Ω（8.01〜8.20、真の値 8）、Ke = 0.1606（0.1603〜0.1609、真の値 0.161）、相関 −0.45）
R, Rlo, Rhi = (num(p, out_dc) for p in [r"R  = ([\d.]+) Ω", r"R  = [\d.]+ Ω（95% 区間 ([\d.]+)〜", r"R  = [\d.]+ Ω（95% 区間 [\d.]+〜([\d.]+)）"])
Ke = num(r"Ke = ([\d.]+) V·s/rad", out_dc)
rho = num(r"I と ω の相関 ([+-][\d.]+)", out_dc)
ok(len(re.findall(r"S0\d\.csv", out_dc)) == 8, "8つのセットアップから、デューティ 100% の RUN が1つずつ（8点）")
ok(abs(R - 8.11) < 0.01 and abs(Rlo - 8.01) < 0.01 and abs(Rhi - 8.20) < 0.01, f"R = {R} Ω（95% 区間 {Rlo}〜{Rhi}。本文 8.11、8.01〜8.20）")
print(f"  ※ 真の値 8 Ω は区間のわずかに外（{Rlo}〜{Rhi}）。区間は8点のばらつきだけから出したもので、"
      "合成データの作り方（1 ms ごとの計算、INA219 の約 9 ms ごとの値）による小さな系統のずれは含まない。実機でも、区間は系統誤差を含まないと考える")
ok(abs(Ke - 0.1606) < 0.0002, f"Ke = {Ke} V·s/rad（本文 0.1606、真の値 0.161）")
ok(abs(rho - (-0.45)) < 0.01, f"I と ω の相関 {rho:+.2f}（本文 −0.45。±1 から遠いので R と Ke を分けられる）")

比べのために、**全部のデューティの RUN** の電池側の電流で $u\,V_{\mathrm{bus}} = R\,i + K_e\,\omega$（$u$ はデューティ）を当てはめてみます。本文 4-7 で「意味のない値になる」と書いた計算です（本文にはコードがないので、ノートブックで追加。回らなかった RUN（5 rpm 未満）は除き、`dcfit` と同じく切片なしの最小二乗）。INA219 の値は PWM のオフの間に D1 を回る電流を含まないのでモータの電流ではなく、不連続導通のためモータの平均の電圧も $uV$ ではないので、式そのものが成り立ちません。

In [ ]:
def fit_all_duties(files):
    """全部のデューティの RUN で u·V_bus = R·i + Ke·ω を当てはめる（誤った使い方の例）"""
    tab = pd.concat([summarize_sweep(pd.read_csv(f), CPR) for f in files])
    tab = tab[tab["rpm"] > 5]
    X = np.c_[tab["i_ma"] / 1000, tab["rpm"] * 2 * np.pi / 60]
    y = tab["duty_set"] / 1000 * tab["v_bus"]
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta, len(tab)


(bR, bK), n1 = fit_all_duties(["data_synth/sweep.csv"])
print(f"合成のスイープ（無負荷・0〜100%、{n1} RUN）：R = {bR:.1f} Ω、Ke = {bK:.3f}")
(fR, fK), n2 = fit_all_duties(sorted(glob.glob("data_synth/fact/S*.csv")))
print(f"要因計画の8区の全部のデューティ（{n2} RUN）：R = {fR:.2f} Ω、Ke = {fK:.3f}")
ok(abs(bR - (-70)) < 1 and abs(bK - 0.29) < 0.005, f"スイープでは R = {bR:.1f} Ω、Ke = {bK:.3f}（本文 −70 Ω、0.29。負の抵抗で意味がない）")
ok(abs(fR - 2.65) < 0.01 and abs(fK - 0.141) < 0.001, f"8区の全部のデューティでは R = {fR:.2f} Ω、Ke = {fK:.3f}（本文 2.65 Ω、0.141。真の値 8・0.161 からずれる）")

## おわりに

- 答えのわかった合成データで、4-7 の「結果の例」の合成データの数値が、本文の4つのプログラムからそのまま出ることを確かめました。
- 回転数がデューティに比例しないのは、PWM のオフの間にコイルの電流が途切れる不連続導通のためで、故障ではありません（4-3-8）。PWM の周波数を上げると、その効き方が変わります。
- 時定数は、10 ms の差分より、カウントそのものを観測とする状態空間モデルのほうが真の値に近く出ました。curve_fit が1回ごとに出す区間は、くり返しのばらつきよりずっと狭いので、「ステップ1回」を独立な単位にして平均と標準偏差で報告します。
- 変えにくい因子（V・L）の効果を調べるときの独立な単位は、セットアップを作り直した回数です。すべての行を独立とみなす分析は、F 値を大きく、p 値を小さく見せます（疑似反復）。
- R・K_e は、INA219 の値がモータの電流と等しいデューティ 100% の RUN だけで求めます。
- このノートブックに入れていないもの：回路を組むこと、Pico B への書き込み（C SDK）、`motor_run.py` での記録、テスタでの確かめ、Pi の `sigrok-cli`。これらは教科書の手順のとおりに行います。